# 03_feature_engineering.ipynb - Fase 2: Feature Engineering

**Objetivo**: Calcular ~20 indicadores OHLCV para cada ticker admitido.

Flujo:
1. 01_eda.ipynb → EDA
2. batch_calibrate.py → Csl + BP
3. **03_feature_engineering.ipynb ← ESTE**
4. 04_labeling.ipynb → Triple Barrier
5. 05_strategy_builder.ipynb → RF + Tree → Reglas
6. 06_backtest_strategies.ipynb → Backtest + Validación

## 1. Imports

In [1]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

from config.settings import DATA, FEATURES, RISK
from src.data import load_ohlc_from_yfinance
from src.features import add_all_features

print("Imports listos")

Imports listos


## 2. Cargar Universo Admitido

In [2]:
admitted_df = pd.read_csv(REPO_ROOT / "data" / "universe_admitted.csv")
print(f"Tickers admitidos: {len(admitted_df)}")
admitted_df.to_string(index=False)

Tickers admitidos: 9


'ticker      csl   bp_median      bp_max  n_samples\n  MCRB 2.073369 1648.271631 2910.751651        500\n  LRMR 2.186679 1735.008416 2948.322504        500\n  ABSI 2.297152 1530.309591 2674.816278        500\n  IMUX 2.348268 1550.750121 2791.049794        500\n  KPTI 2.387959 1318.524144 2546.854384        500\n  HBIO 2.413816 1507.814490 2619.348096        500\n  HOTH 2.462243 1478.604231 2582.297380        500\n  SGMO 2.540506  915.892752 1721.758859        500\n  EBON 2.746795 1192.554669 1985.683930        500'

## 3. Calcular Features para Todos los Tickers

In [3]:
# Este notebook calcula los indicadores y guarda los datasets de features en disco.
features_dir = REPO_ROOT / "data" / "features"
features_dir.mkdir(parents=True, exist_ok=True)

for _, row in admitted_df.iterrows():
    ticker = row['ticker']
    print(f"\nProcesando {ticker}...")

    df = load_ohlc_from_yfinance(ticker, period=DATA.period, interval=DATA.interval)
    df = add_all_features(df)

    feature_cols = [c for c in df.columns if c not in ['Datetime', 'Open', 'High', 'Low', 'Close', 'Volume']]
    print(f"  Barras: {len(df)}")
    print(f"  Features: {feature_cols}")

    feature_path = features_dir / f"features_{ticker}.csv"
    df.to_csv(feature_path, index=False)
    print(f"  Guardado: {feature_path}")


Procesando MCRB...


  Barras: 5049
  Features: ['RSI_14', 'ATR_50', 'EMA_20', 'EMA_50', 'dist_ema20_pct', 'dist_ema50_pct', 'VWAP_20', 'dist_vwap_pct', 'MACD_line', 'MACD_signal', 'MACD_hist', 'RVOL_20', 'ER_Kaufman_10', 'Gap_pct', 'ADX_14', 'ROC_10', 'Momentum_10', 'Volatility_20', 'SMA_10', 'dist_bb_upper_pct', 'dist_bb_lower_pct', 'Candle_Body', 'Candle_Range', 'Upper_Wick', 'Lower_Wick', 'SMA', 'ATR', 'ER', 'RVOL', 'VWAP', 'ROC', 'Estructura_OK']

Procesando LRMR...
  Barras: 5073
  Features: ['RSI_14', 'ATR_50', 'EMA_20', 'EMA_50', 'dist_ema20_pct', 'dist_ema50_pct', 'VWAP_20', 'dist_vwap_pct', 'MACD_line', 'MACD_signal', 'MACD_hist', 'RVOL_20', 'ER_Kaufman_10', 'Gap_pct', 'ADX_14', 'ROC_10', 'Momentum_10', 'Volatility_20', 'SMA_10', 'dist_bb_upper_pct', 'dist_bb_lower_pct', 'Candle_Body', 'Candle_Range', 'Upper_Wick', 'Lower_Wick', 'SMA', 'ATR', 'ER', 'RVOL', 'VWAP', 'ROC', 'Estructura_OK']

Procesando ABSI...
  Barras: 5073
  Features: ['RSI_14', 'ATR_50', 'EMA_20', 'EMA_50', 'dist_ema20_pct', 'dis

## 4. Estadísticas de Features (resumen)

In [4]:
# Placeholder: Mostrar estadísticas de features clave
# Ejemplo para un ticker de muestra

ticker_demo = admitted_df['ticker'].iloc[0]
df_demo = load_ohlc_from_yfinance(ticker_demo, period=DATA.period, interval=DATA.interval)
df_demo = add_all_features(df_demo)

print(f"\nEstadísticas para {ticker_demo}:")
feature_cols = [c for c in df_demo.columns if c not in ['Datetime', 'Open', 'High', 'Low', 'Close', 'Volume']]
print(df_demo[feature_cols].describe())


Estadísticas para MCRB:
            RSI_14       ATR_50       EMA_20       EMA_50  dist_ema20_pct  \
count  5049.000000  5049.000000  5049.000000  5049.000000     5049.000000   
mean     47.795510     0.634863    19.706835    19.968688       -0.507812   
std      17.326240     0.453120    14.768826    15.221359        5.424613   
min       0.000000     0.140426     5.794642     6.140530      -33.014523   
25%      35.860052     0.349800    13.777681    13.686927       -2.957057   
50%      47.753786     0.500600    16.515756    16.653078       -0.604901   
75%      59.612998     0.826080    20.778058    20.574525        2.108568   
max      96.730820     3.025840    99.269742    97.441257       30.463824   

       dist_ema50_pct      VWAP_20  dist_vwap_pct    MACD_line  MACD_signal  \
count     5049.000000  5049.000000    5049.000000  5049.000000  5049.000000   
mean        -1.296234    19.757490      -0.610940    -0.121943    -0.122007   
std          8.401598    14.850623       5.6